# Etapa 1: Preparação dos dados e protocolo de avaliação

Nesta etapa definimos os conjuntos e o pré-processamento, sem ajustar transformadores ou treinar modelos.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## 1.1 Carregamento

O CSV está em `data/dados.csv`. O caminho abaixo funciona com o kernel iniciado na raiz do projeto ou na pasta `notebooks`. Nenhuma estatística é aprendida nesta leitura.

In [ ]:
data_path = Path("data/dados.csv")
if not data_path.is_file():
    data_path = Path("../data/dados.csv")
if not data_path.is_file():
    raise FileNotFoundError("Execute a partir da raiz do projeto ou de notebooks: data/dados.csv não encontrado.")

df_raw = pd.read_csv(data_path)

#Legenda para as colunas:

* 'Age': Idade do funcionário.

* 'Attrition': Desligamento (Indica se o funcionário saiu alguma vez da empresa)

* 'BusinessTravel': Indica a frequência com que o funcionário viaja a travalho

* 'DailyRate': Valor do salário por dia.

* 'Department': Departamento no qual o funcionário trabalha.

* 'DistanceFromHome': Distância em km/milhas da casa ao trabalho.

* 'Education': Nível de escolaridade

* 'EducationField': Área de formação.

* 'EmployeeCount': Coluna constante nos valores observados do projeto original (1), com ausências. Será removida; não deve ser preenchida usando Attrition.

* 'EmployeeNumber': Número de identificação do funcionário; será removido das features.

* 'EnvironmentSatisfaction': Satisfação com o ambiente de trabalho.

* 'Gender': Gênero do funcionário.

* 'HourlyRate': Valor do salário por hora. Observações: Verificar se é proprocional ou não a coluna 'DailyRate' (Valor do salário por dia). Caso haja algum tipo de proporção, basta excluir uma das colunas (informação redundante).

* 'JobInvolvement': Nível de engajamento com as tarefas.

* 'JobLevel': Nível do cargo.

* 'JobRole': Cargo/Função (Descrição do cargo)

* 'JobSatisfaction': Satisfação com o Trabalho (Nível de satisfação geral com o cargo).

* 'MaritalStatus': Estado Civil (Solteiro, Casado, Divorciado).

* 'MonthlyIncome': Renda Mensal (Salário bruto mensal).

* 'MonthlyRate': Valor/Taxa Mensal (Uma taxa interna da empresa, não é o salário).

* 'NumCompaniesWorked': Número de Empresas Anteriores (Quantas empresas o funcionário trabalhou antes desta).

* 'Over18': Acima de 18 Anos (Indica se o funcionário é maior de idade).

* 'OverTime': Horas Extras (Indica se o funcionário faz horas extras - Sim/Não).

* 'PercentSalaryHike': Aumento Percentual do Salário (Percentual do último aumento salarial).

* 'PerformanceRating': Avaliação de Desempenho (Nota da última avaliação de performance).

* 'RelationshipSatisfaction': Satisfação com Relacionamentos (Nível de satisfação com colegas e gestores).

* 'StandardHours': Horas Padrão (A carga horária padrão de trabalho, ex: 80 horas).

* 'StockOptionLevel': Nível de Opção de Ações (Nível do benefício de compra de ações da empresa).

* 'TotalWorkingYears': Total de Anos Trabalhados (Tempo total de experiência de carreira).

* 'TrainingTimesLastYear': Treinamentos no Último Ano (Número de treinamentos que participou no ano anterior).

* 'WorkLifeBalance': Equilíbrio Vida-Trabalho (Nível de satisfação com o equilíbrio entre vida pessoal e profissional).

* 'YearsAtCompany': Anos na Empresa (Tempo total de casa).

* 'YearsInCurrentRole': Anos no Cargo Atual (Tempo que está na função atual).

* 'YearsSinceLastPromotion': Anos Desde a Última Promoção.

* 'YearsWithCurrManager': Anos com o Gestor Atual.

## 1.2 Remoção de colunas previamente justificadas

Preservamos a decisão documentada no projeto original: remover `StandardHours` (80), `Over18` (Y), `EmployeeCount` (1 nos valores observados) e `EmployeeNumber` (identificador). A lista é fixa; não fazemos nova seleção de features usando a base completa.

In [ ]:
columns_to_drop = ["StandardHours", "Over18", "EmployeeCount", "EmployeeNumber"]
df_features = df_raw.drop(columns=columns_to_drop)

## 1.3 Features e alvo

`X` contém apenas as features; `y` representa `Attrition`, com `No = 0` e `Yes = 1`. Validamos ausências e rótulos inesperados antes da conversão. Valores já codificados em 0/1 também são aceitos, sem modificar o alvo original.

In [ ]:
X = df_features.drop(columns="Attrition").copy()
target = df_features["Attrition"]
if target.isna().any() or not target.isin(["No", "Yes", 0, 1]).all():
    raise ValueError("Attrition deve conter apenas No/Yes ou 0/1, sem valores ausentes.")
y = target.map({"No": 0, "Yes": 1, 0: 0, 1: 1}).astype("int64")

## 1.4 Separação imediata entre treino e teste

**Data leakage** ocorre quando informações indisponíveis ao treinamento, como as do teste, influenciam a preparação ou a escolha do modelo. Por isso, separamos os dados antes de aprender medianas, modas ou parâmetros de escala.

`stratify=y` preserva aproximadamente a proporção das classes em cada conjunto. O teste ficará reservado até a avaliação final, sem orientar análises exploratórias, imputação, hiperparâmetros ou limiares. Esta divisão organiza a V2; não torna a base inédita, pois ela já foi explorada no projeto original.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

### Inspeção inicial da base de treinamento

Recuperamos somente as linhas de treino em `df_raw`, pelos índices de `X_train`, para inspecionar também as colunas já removidas das features. Esta cópia é exclusivamente descritiva: não modifica `X_train`, não usa o teste e mantém as ausências.

In [ ]:
df_train_raw = df_raw.loc[X_train.index].copy()
df_train_raw.info()
df_train_raw.head()

### Verificação das colunas removidas no treino

Confirmamos que `StandardHours`, `Over18` e `EmployeeCount` têm um único valor observado, desconsiderando ausências. Verificamos também a completude e a unicidade de `EmployeeNumber`, removido por ser identificador. A decisão de excluir as quatro colunas das features permanece; não preenchemos valores ausentes.

In [ ]:
constant_columns = ["StandardHours", "Over18", "EmployeeCount"]
for column in constant_columns:
    observed_values = df_train_raw[column].dropna().unique()
    print(f"{column}: valores observados = {observed_values}")
    assert len(observed_values) == 1, f"{column} não é constante nos valores observados do treino."

assert df_train_raw["EmployeeNumber"].notna().all(), "Há identificadores ausentes no treino."
assert df_train_raw["EmployeeNumber"].is_unique, "Há identificadores repetidos no treino."

removed_columns_audit = pd.DataFrame({
    "Valores distintos observados": df_train_raw[columns_to_drop].nunique(dropna=True),
    "Valores ausentes": df_train_raw[columns_to_drop].isna().sum(),
})
removed_columns_audit

### Correlação entre MonthlyRate, HourlyRate e DailyRate no treino

Retomamos a hipótese exploratória de que essas taxas poderiam conter informação linear redundante. A matriz de Pearson e o heatmap usam somente `X_train`, com pares de valores disponíveis: não há imputação nem exclusão global de linhas.

In [ ]:
rate_columns = ["MonthlyRate", "HourlyRate", "DailyRate"]
rate_correlation = X_train[rate_columns].corr(method="pearson")
print(rate_correlation.round(6))

fig_rates, ax_rates = plt.subplots(figsize=(6, 4))
sns.heatmap(
    rate_correlation, annot=True, cmap="coolwarm", fmt=".3f",
    vmin=-1, vmax=1, center=0, ax=ax_rates,
)
ax_rates.set_title("Correlação entre taxas — somente treino")
fig_rates.tight_layout()
plt.show()

Nesta divisão de treinamento, as correlações de Pearson são aproximadamente **0,0002** (MonthlyRate × HourlyRate), **−0,0508** (MonthlyRate × DailyRate) e **0,0238** (HourlyRate × DailyRate). Os valores próximos de zero não sustentam a hipótese de forte redundância linear entre essas taxas; portanto, mantemos as três features. Isso não demonstra ausência de relações não lineares nem garante valor preditivo.

## 1.5 Tipos de features no treinamento

Identificamos as colunas somente em `X_train`. Nesta etapa, as escalas ordinais armazenadas como números continuam no grupo numérico.

In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "string", "bool"]).columns.tolist()

unsupported_features = set(X_train.columns) - set(numeric_features) - set(categorical_features)
if unsupported_features:
    raise ValueError(f"Tipos de features não tratados: {sorted(unsupported_features)}")

## 1.6 Fábrica de preprocessadores independentes

Cada chamada cria novos imputadores e transformadores. Numéricas recebem mediana e, com `scale_numeric=True`, padronização; categóricas recebem moda e one-hot encoding com categorias desconhecidas ignoradas. A função apenas constrói o preprocessador: suas estatísticas serão aprendidas posteriormente dentro de cada fold de treinamento, em um pipeline completo.

In [ ]:
def make_preprocessor(numeric_features, categorical_features, scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(steps=numeric_steps)
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer(transformers=[
        ("num", numeric_pipeline, list(numeric_features)),
        ("cat", categorical_pipeline, list(categorical_features)),
    ])

## 1.7 Validação cruzada no treino

A validação estratificada de cinco folds será usada posteriormente apenas com `X_train` e `y_train`. Cada fold ajustará seu próprio pipeline no subconjunto de treinamento e avaliará no de validação. Definir `cv` não treina nem avalia modelos.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 1.8 Base exclusiva para análise exploratória

As seções exploratórias existentes passam a usar `df`, uma cópia somente do treino. Mantemos os rótulos No/Yes nessa cópia para compatibilidade com os gráficos, sem alterar `y_train`. As ausências permanecem nos dados; nenhuma imputação é executada aqui.

In [ ]:
df = X_train.copy()
df["Attrition"] = y_train.map({0: "No", 1: "Yes"})

### Ausências no treinamento

A contagem abaixo é apenas descritiva. A imputação será aprendida pelos pipelines, sem consultar o teste.

In [ ]:
missing_train = X_train.isna().sum()
print(missing_train[missing_train > 0])

### Integridade da separação

Verificamos a ausência do alvo e das colunas removidas nas features, o alinhamento dos índices e a separação entre treino e teste. Estas verificações não ajustam transformadores nem avaliam modelos.

In [ ]:
assert "Attrition" not in X.columns
assert not set(columns_to_drop).intersection(X.columns)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(X)
assert df.index.equals(X_train.index)

A preparação termina com dados ainda não imputados e um protocolo de avaliação definido. As análises seguintes usam exclusivamente o treino; suas saídas antigas foram removidas para não representar a base completa como se fosse este novo subconjunto.

#Parte 2: Visualização e interpretação dos dados:

## 2.1 - Análise Univariada:

O objetivo aqui é examinar apenas uma variavel de cada vez, não queremos nesse primeiro momento encontrar relações ou causas, mas sim descrever cada uma das variáveis presentes no dataset. A análise univariada visa também verificar se as features (variáveis) são consistestes, isto é, se os dados fazem sentido como por exemplo se não existe "idades negativas" no dataset.

### 2.1.1 - Variáveis Categóricas

Se tratando de variaveis categóricas, o foco está em mostrar a contagem e a proporção (ou popularidade) de cada categoria presente na variável. Para isso, um simples gráfico de barras de contagem é suficiente para mostrar todos interesses desse tipo de variável.

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))

sns.countplot(x='Attrition', data=df, ax=ax[0,0]).set_title("Attrition")
ax[0, 0].set_xlabel("")

sns.countplot(x='BusinessTravel', data=df, ax=ax[0, 1]).set_title("BusinessTravel")
ax[0, 1].set_xlabel("")

sns.countplot(x='Department', data=df, ax=ax[1, 0]).set_title("Department")
ax[1, 0].set_xlabel("")

sns.countplot(x='Education', data=df, ax=ax[1, 1]).set_title("Education")
ax[1, 1].set_xlabel("")

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(24, 14))

sns.countplot(x='EducationField', data=df, ax=ax[0,0]).set_title("EducationField")
ax[0, 0].set_xlabel("")

sns.countplot(x='EnvironmentSatisfaction', data=df, ax=ax[0, 1]).set_title("EnvironmentSatisfaction")
ax[0, 1].set_xlabel("")

sns.countplot(x='Gender', data=df, ax=ax[1, 0]).set_title("Gender")
ax[1, 0].set_xlabel("")

sns.countplot(x='JobInvolvement', data=df, ax=ax[1, 1]).set_title("JobInvolvement")
ax[1, 1].set_xlabel("")

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))

sns.countplot(x='JobLevel', data=df, ax=ax[0,0]).set_title("JobLevel")
ax[0, 0].set_xlabel("")

sns.countplot(x='JobSatisfaction', data=df, ax=ax[0, 1]).set_title("JobSatisfaction")
ax[0, 1].set_xlabel("")

sns.countplot(x='MaritalStatus', data=df, ax=ax[1, 0]).set_title("MaritalStatus")
ax[1, 0].set_xlabel("")

sns.countplot(x='PerformanceRating', data=df, ax=ax[1, 1]).set_title("PerformanceRating")
ax[1, 1].set_xlabel("")

In [ ]:
fig, ax = plt.subplots(nrows=4, ncols=1, figsize=(15, 30))

sns.countplot(x='RelationshipSatisfaction', data=df, ax=ax[0]).set_title("RelationshipSatisfaction")
ax[0].set_xlabel("")

sns.countplot(x='StockOptionLevel', data=df, ax=ax[1]).set_title("StockOptionLevel")
ax[1].set_xlabel("")

sns.countplot(x='WorkLifeBalance', data=df, ax=ax[2]).set_title("WorkLifeBalance")
ax[2].set_xlabel("")

sns.countplot(y = 'JobRole', data=df, ax=ax[3]).set_title("Count")
ax[3].set_xlabel("")


### 2.1.2 - Variaveis Numéricas

Para variaveis numéricas, o foco está em mostrar o "centro" dos dados (medidas de tendência central), o quão espalhados estão os dados (dispersão ou variância) e forma dos dados (distribuição e a presença de outliers). Para isso, vamos utilizar histogramas junto a média e mediana.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def plotar_univariada_numerica(df, column_name, ax_hist, ax_box, bins):
    """
    Plota um histograma (com média/mediana) e um boxplot lado a lado
    para uma coluna numérica de um DataFrame.

    Parâmetros:
    - df (pd.DataFrame): O DataFrame contendo os dados.
    - column_name (str): O nome da coluna a ser plotada.
    - ax_hist (matplotlib.axis): O eixo onde o histograma será desenhado.
    - ax_box (matplotlib.axis): O eixo onde o boxplot será desenhado.
    - bins (int, opcional): O número de bins (barras) para o histograma.
    """

    if not pd.api.types.is_numeric_dtype(df[column_name]):
        print(f"Aviso: Coluna '{column_name}' não é numérica. O gráfico pode falhar.")
        return

    media = df[column_name].mean()
    mediana = df[column_name].median()

    sns.histplot(data=df, x=column_name, ax=ax_hist, bins=bins, kde=True)

    ax_hist.axvline(media, color='red', linestyle='--', linewidth=2, label=f'Média: {media:.2f}')
    ax_hist.axvline(mediana, color='green', linestyle='-', linewidth=2, label=f'Mediana: {mediana:.2f}')

    ax_hist.legend()
    ax_hist.set_title(f'Distribuição de {column_name}')
    ax_hist.set_xlabel(column_name)
    ax_hist.set_ylabel('Frequência')

    sns.boxplot(data=df, y=column_name, ax=ax_box)
    ax_box.set_title(f'Boxplot de {column_name}')
    ax_box.set_ylabel(column_name)

In [ ]:
print(df.columns)

In [ ]:
fig, eixos = plt.subplots(nrows=14, ncols=2, figsize=(12 ,44))

plotar_univariada_numerica(df, 'Age',ax_hist=eixos[0, 0], ax_box=eixos[0, 1], bins=7)

plotar_univariada_numerica(df, 'DailyRate',ax_hist=eixos[1, 0], ax_box=eixos[1, 1], bins=7)

plotar_univariada_numerica(df, 'DistanceFromHome',ax_hist=eixos[2, 0], ax_box=eixos[2, 1], bins=7)

plotar_univariada_numerica(df, 'HourlyRate',ax_hist=eixos[3, 0], ax_box=eixos[3, 1], bins=7)

plotar_univariada_numerica(df, 'MonthlyIncome',ax_hist=eixos[4, 0], ax_box=eixos[4, 1], bins=7)

plotar_univariada_numerica(df, 'MonthlyRate',ax_hist=eixos[5, 0], ax_box=eixos[5, 1], bins=7)

plotar_univariada_numerica(df, 'NumCompaniesWorked',ax_hist=eixos[6, 0], ax_box=eixos[6, 1], bins=9)

plotar_univariada_numerica(df, 'PercentSalaryHike',ax_hist=eixos[7, 0], ax_box=eixos[7, 1], bins=7)

plotar_univariada_numerica(df, 'TotalWorkingYears',ax_hist=eixos[8, 0], ax_box=eixos[8, 1], bins=7)

plotar_univariada_numerica(df, 'TrainingTimesLastYear',ax_hist=eixos[9, 0], ax_box=eixos[9, 1], bins=7)

plotar_univariada_numerica(df, 'YearsAtCompany',ax_hist=eixos[10, 0], ax_box=eixos[10, 1], bins=7)

plotar_univariada_numerica(df, 'YearsInCurrentRole',ax_hist=eixos[11, 0], ax_box=eixos[11, 1], bins=7)

plotar_univariada_numerica(df, 'YearsSinceLastPromotion',ax_hist=eixos[12, 0], ax_box=eixos[12, 1], bins=7)

plotar_univariada_numerica(df, 'YearsWithCurrManager',ax_hist=eixos[13, 0], ax_box=eixos[13, 1], bins=7)

plt.tight_layout() # Ajusta o espaçamento
plt.savefig('Variaveis_num.png')
plt.show()


## 2.2 - Análise Bivariada:

Na análise bivariada, o objetivo é examinar a relação entre duas variáveis simultaneamente. Em vez de apenas descrever uma variável isolada, buscamos ativamente por associações, correlações ou dependências entre elas.

Para um modelo de previsão, o insight fundamental é descobrir se uma feature (seja ela categórica ou uma faixa de valor) tem poder preditivo sobre a nossa variável-alvo (o attrition)

A escolha do gráfico para fazer isso depende dos tipos de variáveis. O gráfico de barras agrupado (ou empilhado), por exemplo, é excelente para comparar duas variáveis categóricas (como Department vs. Attrition). No entanto, para analisar uma variável numérica contra o attrition (como Idade vs. Attrition), o ideal seria boxplots ou violin plots.

###2.2.1 - Variáveis Categóricas:



In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))

sns.countplot(x='JobSatisfaction', hue='Attrition', data=df, ax=ax[0, 0], hue_order=['No', 'Yes']).set_title("JobSatisfaction")
ax[0, 0].set_xlabel("")

sns.countplot(x='BusinessTravel', hue='Attrition', data=df, ax=ax[0, 1], hue_order=['No', 'Yes']).set_title("BusinessTravel")
ax[0, 1].set_xlabel("")

sns.countplot(x='Department', hue='Attrition', data=df, ax=ax[1, 0], hue_order=['No', 'Yes']).set_title("Department")
ax[1, 0].set_xlabel("")

sns.countplot(x='Education', hue='Attrition', data=df, ax=ax[1, 1], hue_order=['No', 'Yes']).set_title("Education")
ax[1, 1].set_xlabel("")

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(22, 8))
sns.countplot(x='EducationField', hue='Attrition', data=df, ax=ax[0,0], hue_order=['No', 'Yes']).set_title("EducationField")
ax[0, 0].set_xlabel("")

sns.countplot(x='EnvironmentSatisfaction', hue='Attrition', data=df, ax=ax[0, 1], hue_order=['No', 'Yes']).set_title("EnvironmentSatisfaction")
ax[0, 1].set_xlabel("")

sns.countplot(x='Gender', hue='Attrition', data=df, ax=ax[1, 0], hue_order=['No', 'Yes']).set_title("Gender")
ax[1, 0].set_xlabel("")

sns.countplot(x='JobInvolvement', hue='Attrition', data=df, ax=ax[1, 1], hue_order=['No', 'Yes']).set_title("JobInvolvement")
ax[1, 1].set_xlabel("")

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))
sns.countplot(x='RelationshipSatisfaction', hue='Attrition', data=df, ax=ax[0,0], hue_order=['No', 'Yes']).set_title("RelationshipSatisfaction")
ax[0,0].set_xlabel("")

sns.countplot(x='JobLevel', hue='Attrition', data=df, ax=ax[0,1], hue_order=['No', 'Yes']).set_title("JobLevel")
ax[0, 1].set_xlabel("")

sns.countplot(x='MaritalStatus', hue='Attrition', data=df, ax=ax[1, 0], hue_order=['No', 'Yes']).set_title("MaritalStatus")
ax[1, 0].set_xlabel("")

sns.countplot(x='PerformanceRating', hue='Attrition', data=df, ax=ax[1, 1], hue_order=['No', 'Yes']).set_title("PerformanceRating")
ax[1, 1].set_xlabel("")


In [ ]:
fig, ax = plt.subplots(nrows=3, ncols=1, figsize=(14, 12))

sns.countplot(x='StockOptionLevel', hue='Attrition', data=df, ax=ax[0], hue_order=['No', 'Yes']).set_title("StockOptionLevel")
ax[0].set_xlabel("")

sns.countplot(x='WorkLifeBalance', hue='Attrition', data=df, ax=ax[1], hue_order=['No', 'Yes']).set_title("WorkLifeBalance")
ax[1].set_xlabel("")

sns.countplot(data=df, x='JobRole', hue='Attrition', ax=ax[2], hue_order=['No', 'Yes'])
ax[2].set_title('JobRole')
ax[2].set_xlabel("")
ax[2].tick_params(axis='x', rotation=90)


### 2.2.2 - Variáveis Numéricas

In [ ]:
fig, ax_uni = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))

sns.histplot(data=df, x='Age', hue='Attrition', ax=ax_uni[0, 0], hue_order=['No', 'Yes'])
ax_uni[0, 0].set_title('Age')
ax_uni[0,0].set_xlabel("")

sns.histplot(data=df, x='DailyRate', hue='Attrition', ax=ax_uni[0, 1], hue_order=['No', 'Yes'])
ax_uni[0, 1].set_title('DailyRate')
ax_uni[0, 1].set_xlabel("")

sns.histplot(data=df, x='DistanceFromHome', hue='Attrition', ax=ax_uni[1, 0], hue_order=['No', 'Yes'])
ax_uni[1, 0].set_title('DistanceFromHome')
ax_uni[1, 0].set_xlabel("")

sns.histplot(data=df, x='HourlyRate', hue='Attrition', ax=ax_uni[1, 1], hue_order=['No', 'Yes'])
ax_uni[1, 1].set_title('HourlyRate')
ax_uni[1, 1].set_xlabel("")

In [ ]:
fig, ax_uni = plt.subplots(nrows=2, ncols=2, figsize=(22, 10))

sns.histplot(data=df, x='MonthlyIncome', hue='Attrition', ax=ax_uni[0, 0], hue_order=['No', 'Yes'])
ax_uni[0, 0].set_title('MonthlyIncome')
ax_uni[0, 0].set_xlabel("")

sns.histplot(data=df, x='MonthlyRate', hue='Attrition', ax=ax_uni[0, 1], hue_order=['No', 'Yes'])
ax_uni[0, 1].set_title('MonthlyRate')
ax_uni[0, 1].set_xlabel("")

sns.histplot(y='NumCompaniesWorked', hue='Attrition', data=df, ax=ax_uni[1, 0], hue_order=['No', 'Yes']).set_title("NumCompaniesWorked")
ax_uni[1, 0].set_title('NumCompaniesWorked')
ax_uni[1, 0].set_ylabel("")

sns.histplot(data=df, x='Age', hue='Attrition', ax=ax_uni[1, 1], hue_order=['No', 'Yes'])
ax_uni[1, 1].set_title('Age')
ax_uni[1,1].set_xlabel("")

plt.subplots_adjust(hspace=0.9)

In [ ]:
fig, ax_uni = plt.subplots(nrows=2, ncols=1, figsize=(14, 10))

sns.histplot(data=df, x='DailyRate', hue='Attrition', ax=ax_uni[0], hue_order=['No', 'Yes'])
ax_uni[0].set_title('DailyRate')
ax_uni[0].set_xlabel("")

sns.histplot(data=df, x='DistanceFromHome', hue='Attrition', ax=ax_uni[1], hue_order=['No', 'Yes'])
ax_uni[1].set_title('DistanceFromHome')
ax_uni[1].set_xlabel("")

# Parte 3: Análise de correlação no treino

Esta análise usa apenas `df`, derivado de `X_train`. Na codificação exploratória, ausências categóricas recebem indicadores (`dummy_na=True`); ausências numéricas permanecem e as correlações usam pares disponíveis. Não há imputação nem exclusão global de linhas. Essa codificação é apenas para exploração e não alimenta os modelos.

Vamos começar definindo funções que nos auxiliarão

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from itertools import combinations

#Funções Auxiliares

def filter_correlation_matrix(corr_matrix, prefixes):
    matrix_filtrada = corr_matrix.copy()
    for prefix in prefixes:
        cols_com_prefixo = [col for col in matrix_filtrada.columns if col.startswith(prefix)]
        if len(cols_com_prefixo) > 1:
            for var1, var2 in combinations(cols_com_prefixo, 2):
                matrix_filtrada.loc[var1, var2] = np.nan
                matrix_filtrada.loc[var2, var1] = np.nan
    return matrix_filtrada

def print_top_10(filtered_matrix, method_name):
    corr_pares_filtrados = filtered_matrix.unstack()

    top_10_abs_idx = (
        corr_pares_filtrados
        .abs()
        .dropna()
        .sort_values(ascending=False)
        [lambda s: s < 1.0]
        .drop_duplicates()
        .head(10)
    ).index

    top_10_final = corr_pares_filtrados.loc[top_10_abs_idx]

    df_top_10 = (
        top_10_final
        .to_frame(name='Correlacao')
        .reset_index()
        .rename(columns={'level_0': 'Variável 1', 'level_1': 'Variável 2'})
    )

    print(f"\n--- Top 10 Correlações ({method_name.upper()}) ---")
    print(df_top_10.to_string(index=False))
    print("-" * 70)

def plot_heatmap(filtered_matrix, method_name):
    n_cols = len(filtered_matrix.columns)
    plt.figure(figsize=(max(15, n_cols * 0.4), max(12, n_cols * 0.3)))

    sns.heatmap(
        filtered_matrix,
        cmap='coolwarm',
        vmin=-1,
        vmax=1,
        annot=False
    )

    title = f'Matriz de Correlação {method_name.capitalize()} (Filtrada)'
    filename = f'matriz_corr_{method_name}_filtrada.png'

    plt.title(title, fontsize=16)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()

cols_object = df.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

prefixes = [col + '_' for col in cols_object]

df_encoded = df.pipe(pd.get_dummies, columns=cols_object, drop_first=False, dummy_na=True)

print("DataFrame codificado e pronto. Você já pode rodar os blocos de análise.")

Agora basta utiliza-las para criar a visualização das correlações usando dois métodos: Pearson e Spearman

In [ ]:
print("ANÁLISE DE PEARSON")
matriz_corr_pearson = df_encoded.corr(method='pearson')
matriz_pearson_filtrada = filter_correlation_matrix(matriz_corr_pearson, prefixes)
print_top_10(matriz_pearson_filtrada, 'pearson')
plot_heatmap(matriz_pearson_filtrada, 'pearson')

In [ ]:
print("\nINICIANDO ANÁLISE DE SPEARMAN")
matriz_corr_spearman = df_encoded.corr(method='spearman')
matriz_spearman_filtrada = filter_correlation_matrix(matriz_corr_spearman, prefixes)
print_top_10(matriz_spearman_filtrada, 'spearman')
plot_heatmap(matriz_spearman_filtrada, 'spearman')


Agora vamos ver os pares de variáveis que são considerados bem correlacionados por ambos os métodos:

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from itertools import combinations

def filter_correlation_matrix(corr_matrix, prefixes):
    matrix_filtrada = corr_matrix.copy()
    for prefix in prefixes:
        cols_com_prefixo = [col for col in matrix_filtrada.columns if col.startswith(prefix)]
        if len(cols_com_prefixo) > 1:
            for var1, var2 in combinations(cols_com_prefixo, 2):
                matrix_filtrada.loc[var1, var2] = np.nan
                matrix_filtrada.loc[var2, var1] = np.nan
    return matrix_filtrada

def get_top_10_pairs(corr_matrix, prefixes):
    """
    Usa sua função de filtro e retorna um 'set' com os 10
    maiores pares de correlação.
    """
    matrix_filtrada = filter_correlation_matrix(corr_matrix, prefixes)
    corr_pares_filtrados = matrix_filtrada.unstack()

    top_10_abs_idx = (
        corr_pares_filtrados
        .abs()
        .dropna()
        .sort_values(ascending=False)
        [lambda s: s < 1.0]
        .drop_duplicates()
        .head(10)
    ).index

    #Normaliza os pares (A,B) e (B,A) para {A,B} para comparação
    normalized_pairs = {tuple(sorted(pair)) for pair in top_10_abs_idx}
    return normalized_pairs


cols_object = df.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
prefixes = [col + '_' for col in cols_object]
df_encoded = df.pipe(pd.get_dummies, columns=cols_object, drop_first=False, dummy_na=True)

#Obter pares do Pearson
matriz_corr_pearson = df_encoded.corr(method='pearson')
pares_pearson = get_top_10_pairs(matriz_corr_pearson, prefixes)

#Obter pares do Spearman
matriz_corr_spearman = df_encoded.corr(method='spearman')
pares_spearman = get_top_10_pairs(matriz_corr_spearman, prefixes)

#Intersecção
pares_em_comum = pares_pearson.intersection(pares_spearman)

print("Pares Comuns no Top 10 (Pearson & Spearman):")
print(f"Total de pares encontrados: {len(pares_em_comum)}")

if not pares_em_comum:
    print("Nenhum par em comum encontrado.")
else:
    for i, (var1, var2) in enumerate(pares_em_comum):
        print(f"  {i+1}. {var1}  <-->  {var2}")

Podemos dizer que esse pares de fatos tem uma relação forte, e isso é coerente de fato, por exemplo, "JobLevel" com "TotalWorkingYears", a tendência é alcançar cargos maiores conforme o aumento de tempo de contribuição.

# Parte 4: Comparação inicial por validação cruzada

Esta etapa é **model selection**: comparamos cinco configurações fixas exclusivamente em `X_train` e `y_train`, usando o `StratifiedKFold` já definido. Não fazemos tuning, ajuste de threshold ou escolha de vencedor.

`X_test` e `y_test` continuam reservados. O teste final ocorrerá somente depois da escolha do modelo e das decisões de tuning/threshold, sem orientar essas decisões. Os resultados históricos foram substituídos por esta avaliação; os scores dos folds são de validação, não do teste reservado.

## 4.1 Modelos e referências de comparação

**DummyClassifier:** prevê sempre a classe mais frequente do treinamento de cada fold. É a referência trivial para medir quanto uma accuracy elevada pode refletir apenas o desbalanceamento.

**Regressão Logística:** referência simples, eficiente e adequada à classificação binária. Modela uma combinação linear das features transformadas na escala de log-odds. A padronização auxilia a otimização e a regularização. Sua simplicidade e interpretabilidade devem ser consideradas junto com o desempenho, pelo princípio de parcimônia.

**Random Forest:** combina árvores com aleatoriedade nas amostras e nas features. Pode capturar interações e relações não lineares. Maior complexidade não garante melhor desempenho na classe positiva.

**Gradient Boosting:** constrói árvores sequencialmente, buscando reduzir os erros segundo a função de perda. Sua adequação a dados tabulares é uma motivação para experimentá-lo, não uma garantia de superioridade.

**XGBoost:** outra implementação de boosting de árvores. `scale_pos_weight` pondera a contribuição da classe positiva à perda; não reamostra os dados nem garante aumento de recall. Comparar GBM com XGBoost altera simultaneamente o algoritmo e a ponderação, portanto diferenças observadas não podem ser atribuídas apenas ao peso das classes.

In [1]:
import platform
import numpy as np
import pandas as pd
import sklearn
import xgboost

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from sklearn.metrics import (
    make_scorer, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
)
from xgboost import XGBClassifier

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
})

{'python': '3.11.14', 'numpy': '2.4.6', 'pandas': '3.0.5', 'scikit-learn': '1.9.0', 'xgboost': '3.2.0'}


## 4.2 Configurações fixas e pipelines independentes

Cada modelo recebe uma nova instância de `make_preprocessor`. Apenas a Regressão Logística usa escala; Dummy e árvores usam imputação e one-hot encoding sem `StandardScaler`.

O peso do XGBoost é `n_negativos / n_positivos`, calculado uma única vez com `y_train` e mantido fixo nos folds, conforme este protocolo. Isso usa a distribuição dos rótulos de todo o treino, incluindo as parcelas de validação interna; não é um peso reestimado no treinamento de cada fold. Já `class_weight="balanced"` da Regressão Logística e da Random Forest é calculado pelo estimador a cada ajuste. Nenhum desses pesos utiliza o teste reservado.

A validação cruzada clonará os pipelines: imputadores, encoder e escala serão ajustados somente no treinamento do fold. `n_jobs=1` limita a execução a um processo de validação e uma thread no XGBoost; não representa tuning.

In [2]:
assert not y_train.isna().any()
assert set(y_train.unique()) == {0, 1}
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_configs = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=42,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, random_state=42,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100, random_state=42,
        scale_pos_weight=scale_pos_weight, eval_metric="logloss", n_jobs=1,
    ),
}

model_pipelines = {
    name: Pipeline(steps=[
        ("preprocessor", make_preprocessor(
            numeric_features, categorical_features,
            scale_numeric=(name == "Logistic Regression"),
        )),
        ("classifier", classifier),
    ])
    for name, classifier in model_configs.items()
}
print(f"scale_pos_weight calculado com y_train: {scale_pos_weight:.6f}")

scale_pos_weight calculado com y_train: 5.191011


## 4.3 Métricas

A classe positiva é **1 / Attrition Yes**. Accuracy mede os acertos globais; precision mede a fração de alertas positivos corretos; recall mede a fração de saídas detectadas; F1 é a média harmônica de precision e recall da classe positiva.

ROC-AUC e Average Precision avaliam os scores de probabilidade sem escolher um novo limiar. Average Precision resume a curva precision-recall e deve ser interpretada considerando a prevalência positiva. Em classificação binária com rótulos 0/1, o scorer de ROC-AUC usa a probabilidade da classe 1.

Os scorers de precision, recall e F1 usam `average="binary"`, `pos_label=1` e `zero_division=0`. Assim, a precision do Dummy é registrada como zero quando ele não prevê positivos. Usamos as decisões padrão de `predict`, sem ajustar threshold.

In [3]:
scoring = {
    "accuracy": make_scorer(accuracy_score),
    "precision": make_scorer(
        precision_score, average="binary", pos_label=1, zero_division=0,
    ),
    "recall": make_scorer(
        recall_score, average="binary", pos_label=1, zero_division=0,
    ),
    "f1": make_scorer(
        f1_score, average="binary", pos_label=1, zero_division=0,
    ),
    "roc_auc": make_scorer(roc_auc_score, response_method="predict_proba"),
    "average_precision": make_scorer(
        average_precision_score, response_method="predict_proba", pos_label=1,
    ),
}

## 4.4 Execução da validação cruzada

Reutilizamos `cv`, com cinco folds estratificados, embaralhamento e semente 42. As divisões são as mesmas para todos os modelos. Cada chamada ajusta cinco clones do pipeline; não há ajuste final em todo o treino nesta etapa. Erros interrompem a execução em vez de serem convertidos silenciosamente em scores ausentes.

In [4]:
cv_results = {}
fold_records = []

for model_name, pipeline in model_pipelines.items():
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv, scoring=scoring,
        n_jobs=1, return_train_score=False, error_score="raise",
    )
    cv_results[model_name] = scores
    for fold in range(cv.get_n_splits()):
        fold_records.append({
            "model": model_name,
            "fold": fold + 1,
            **{metric: scores[f"test_{metric}"][fold] for metric in scoring},
        })
    print(f"{model_name}: {cv.get_n_splits()} folds concluídos")

fold_results = pd.DataFrame(fold_records)

Dummy: 5 folds concluídos
Logistic Regression: 5 folds concluídos
Random Forest: 5 folds concluídos
Gradient Boosting: 5 folds concluídos
XGBoost: 5 folds concluídos


## 4.5 Tabela comparativa

`cv_summary` contém média e desvio padrão amostral (`ddof=1`) de cada métrica nos cinco folds, sem ordenar os modelos como um ranking. O desvio padrão descreve a variação entre folds; não é um intervalo de confiança nem comprova significância de diferenças.

As chaves `test_*` retornadas pela API `cross_validate` significam **validação interna do fold**, não acesso ao conjunto de teste reservado.

In [5]:
cv_summary = (
    fold_results.groupby("model", sort=False)[list(scoring)]
    .agg(["mean", "std"])
    .reindex(model_pipelines)
)
cv_summary.columns = [
    f"{metric}_{statistic}" for metric, statistic in cv_summary.columns
]
cv_summary.round(4)

,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
model,,,,,,,,,,,,
Dummy,0.8385,0.0022,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,0.0000,0.1615,0.0022
Logistic Regression,0.7750,0.0258,0.3968,0.0362,0.7529,0.0921,0.5190,0.0498,0.8374,0.0535,0.6273,0.1154
Random Forest,0.8602,0.0199,0.6330,0.1100,0.3592,0.0510,0.4538,0.0560,0.7916,0.0427,0.5337,0.0464
Gradient Boosting,0.8748,0.0091,0.7806,0.0968,0.3143,0.0330,0.4475,0.0457,0.8180,0.0287,0.5727,0.0392
XGBoost,0.8593,0.0153,0.5967,0.0695,0.3875,0.0970,0.4663,0.0881,0.8025,0.0295,0.5509,0.0542


## 4.6 Interpretação dos trade-offs

A leitura abaixo usa as médias efetivamente calculadas. Accuracy deve ser confrontada com o Dummy e com as métricas da classe positiva. Precision e recall descrevem compromissos diferentes: mais alertas podem recuperar mais saídas e também gerar falsos positivos. Os pesos de classe não garantem o mesmo compromisso entre algoritmos.

Nesta execução, a Regressão Logística apresentou recall médio de 0.753 e precision média de 0.397, enquanto o Gradient Boosting apresentou recall de 0.314 e precision de 0.781. A primeira configuração recupera uma fração maior das saídas observadas, mas uma fração menor dos seus alertas positivos está correta. A segunda gera alertas mais precisos, mas deixa de detectar uma fração maior das saídas. A Random Forest, mesmo com ponderação de classes, teve recall médio de 0.359: o uso de pesos não assegura alta detecção da classe minoritária. Essas observações não estabelecem um vencedor.

In [6]:
positive_rate = y_train.mean()
print(f"Prevalência de Attrition Yes no treino: {positive_rate:.2%}.")
dummy_scores = cv_summary.loc["Dummy"]
print(
    f"Dummy: accuracy média {dummy_scores['accuracy_mean']:.3f}, "
    f"recall positivo {dummy_scores['recall_mean']:.3f} e "
    f"F1 positivo {dummy_scores['f1_mean']:.3f}. "
    "A accuracy elevada da referência majoritária não significa detecção de saídas."
)
for model_name in model_pipelines:
    if model_name == "Dummy":
        continue
    values = cv_summary.loc[model_name]
    print(
        f"{model_name}: precision {values['precision_mean']:.3f}, "
        f"recall {values['recall_mean']:.3f}, F1 {values['f1_mean']:.3f}; "
        f"accuracy {values['accuracy_mean']:.3f} "
        f"(diferença para Dummy: {values['accuracy_mean'] - dummy_scores['accuracy_mean']:+.3f}); "
        f"ROC-AUC {values['roc_auc_mean']:.3f}, "
        f"Average Precision {values['average_precision_mean']:.3f}."
    )

Prevalência de Attrition Yes no treino: 16.15%.
Dummy: accuracy média 0.838, recall positivo 0.000 e F1 positivo 0.000. A accuracy elevada da referência majoritária não significa detecção de saídas.
Logistic Regression: precision 0.397, recall 0.753, F1 0.519; accuracy 0.775 (diferença para Dummy: -0.064); ROC-AUC 0.837, Average Precision 0.627.
Random Forest: precision 0.633, recall 0.359, F1 0.454; accuracy 0.860 (diferença para Dummy: +0.022); ROC-AUC 0.792, Average Precision 0.534.
Gradient Boosting: precision 0.781, recall 0.314, F1 0.447; accuracy 0.875 (diferença para Dummy: +0.036); ROC-AUC 0.818, Average Precision 0.573.
XGBoost: precision 0.597, recall 0.387, F1 0.466; accuracy 0.859 (diferença para Dummy: +0.021); ROC-AUC 0.802, Average Precision 0.551.


## 4.7 Limites e próximas decisões

Esta comparação é exploratória e não declara um modelo vencedor. Os scores de validação usados para seleção não substituem uma avaliação final independente. Diferenças entre configurações devem ser consideradas junto com a variação entre folds, a simplicidade e os custos de falsos positivos e falsos negativos.

Em etapas posteriores serão definidas a escolha do modelo e eventuais decisões de tuning/threshold usando somente o treinamento e sua validação interna. Apenas depois dessas decisões o pipeline escolhido poderá ser ajustado em todo o treino e avaliado uma vez em `X_test` e `y_test`. O teste reservado não foi usado nesta Parte 4.

Para reprodução, as versões de Python e das bibliotecas desta execução estão registradas na saída da célula de imports. No macOS, o XGBoost também requer o runtime OpenMP (`libomp`). O notebook não instala dependências automaticamente.